# 🎯 4. Routing & Query Construction

Σε ένα μεγάλο RAG σύστημα έχουμε συχνά:

* **Πολλαπλές πηγές γνώσης** (πχ technical docs, FAQs, billing data)
* **Πολλαπλούς τύπους ερωτήσεων** (πχ συγκεκριμένα facts vs. how-to)
* **Δομημένα metadata** που θέλουμε να χρησιμοποιήσουμε ως filters

Σε αυτό το notebook μαθαίνουμε:

1. **Logical routing** : κατευθύνουμε queries σε διαφορετικές πηγές με structured output
2. **Semantic routing** : επιλογή prompt/strategy βάσει embedding similarity
3. **Query construction** : μετατροπή φυσικής γλώσσας σε δομημένο query με metadata filters

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(__file__).parent / ".env" if "__file__" in dir() else Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

## 4.1 Γιατί χρειαζόμαστε routing

Ας υποθέσουμε ότι η NebulaTech έχει **3 ξεχωριστά indices**:

* `technical_docs` — προδιαγραφές, APIs, configuration
* `pricing_docs` — χρεώσεις, plans, discounts
* `support_docs` — troubleshooting, FAQs

Αν τα ενώσουμε σε ένα τεράστιο index, το vocabulary μπερδεύεται και το embedding similarity γίνεται θόρυβος.

**Λύση:** Ένα **router** που, βάσει της ερώτησης, αποφασίζει σε **ποιο** index θα ψάξει.

```
                  ┌─► technical_index ─► retrieve
Question ──[router]──┼─► pricing_index    ─► retrieve
                  └─► support_index    ─► retrieve
```

<img src="images/routing_architecture.png" width="70%" style="border-radius:10px;margin:12px 0;"/>

***Εικ. 4.1** — Multi-source routing: ο router αποφασίζει ποιο index να ρωτήσει. LLM-based (structured output) ή Embedding-based (cosine similarity).*

## 4.2 Logical Routing με structured output

Χρησιμοποιούμε function-calling του LLM για να επιστρέφει **τύπο** της ερώτησης ως ένα Pydantic enum.

In [2]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

class RouteDecision(BaseModel):
    """Route a user question to the correct DAtanous.ai knowledge source."""
    datasource: Literal["product_docs", "pricing_docs", "support_docs"] = Field(
        ...,
        description= (
            "The knowledge source to use:\n"
            "- 'product_docs' for features, APIs, integrations and architecture\n"
            "- 'pricing_docs' for plans, pricing tiers, discounts and billing\n"
            "- 'support_docs' for troubleshouting, errors, and operational issues"
        ),
    )
    reasoning: str = Field(description="Brief explanation of why this source was chosen.")

llm = ChatOpenAI(model=LLM_MODEL, temperature=0.0)
router_llm = llm.with_structured_output(RouteDecision)

ROUTER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", 
     "You are an expert router for the DAtanous.ai support system."
     "Give user input question, select most appropriate knowledge source:"
     "product_docs, pricing_docs, support_docs"),
     ("human", "{question}")
])

router_chain = ROUTER_PROMPT | router_llm

test_questions = [
    "How does Datanous Guard detect prompt injection?",
    "What is the price of Datanous Insight Professional?",
    "Why is my Datanous Search API returning empty results?",
]

for q in test_questions:
    decision = router_chain.invoke({"question": q})
    print(f"Q: {q}")
    print(f" -> {decision.datasource} | {decision.reasoning[:100]}...")

Q: How does Datanous Guard detect prompt injection?
 -> product_docs | The question pertains to the features and functionalities of Datanous Guard, specifically how it det...
Q: What is the price of Datanous Insight Professional?
 -> pricing_docs | The question pertains to the pricing of a specific product, which falls under the category of pricin...
Q: Why is my Datanous Search API returning empty results?
 -> support_docs | The question pertains to troubleshooting an issue with the Datanous Search API, which is best addres...


### Σύνδεση router με πραγματικές αλυσίδες

## 4.3 Semantic Routing

Άλλος τρόπος επιλογής: αντί να ρωτάμε ένα LLM, μετράμε **embedding similarity** ανάμεσα στην ερώτηση και σε **prompt-templates**. Πιο γρήγορο και φθηνό.

Συχνή χρήση: **επιλογή προσωπικότητας/style** του απαντητή.

In [3]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
 
embedder = OpenAIEmbeddings(model=EMBED_MODEL)
 
# Three focused Chroma stores — one per knowledge domain
product_store = Chroma.from_documents(
    [
        Document(page_content="Datanous Insight ingests PDF, Word, Markdown, HTML, and plain-text formats via a hybrid indexing pipeline."),
        Document(page_content="Datanous Guard is a LangChain-compatible RunnableLambda performing grounding, injection detection, PII redaction, and tenant isolation."),
        Document(page_content="Datanous Embed supports text-embedding-3-small (1536 dims) and text-embedding-3-large (3072 dims) with Redis caching and batch size up to 2048."),
        Document(page_content="Datanous Search exposes a REST API with filter fields: document_type, department, date_range, and language. Returns ranked results in under 200 ms."),
    ],
    embedding=embedder, collection_name="product",
)
 
pricing_store = Chroma.from_documents(
    [
        Document(page_content="Datanous Insight Starter: 50 USD per month, up to 10,000 document pages, single-tenant, community support."),
        Document(page_content="Datanous Insight Professional: 350 USD per month, up to 100,000 pages, multi-tenant row-level access control, email support."),
        Document(page_content="Datanous Insight Enterprise: unlimited pages, dedicated vector store, 99.9 percent uptime SLA, custom pricing, on-premises option."),
        Document(page_content="All Datanous.ai plans include TLS 1.3 encryption in transit and AES-256 at rest. GDPR Data Processing Agreement included from Professional tier."),
    ],
    embedding=embedder, collection_name="pricing",
)
 
support_store = Chroma.from_documents(
    [
        Document(page_content="Empty results from Datanous Search: verify that the corpus has been re-indexed after document updates and that the metadata filter syntax matches the schema."),
        Document(page_content="Datanous Embed returning 429 errors: the batch size exceeds 2048. Split the input list into smaller batches and retry with exponential backoff."),
        Document(page_content="Datanous Guard blocking legitimate queries: review the grounding threshold. The default is 0.7; lowering it to 0.5 reduces false positives."),
        Document(page_content="Datanous Insight Professional showing cross-tenant data: ensure tenant_id is set correctly in the request header and that Guard middleware is enabled."),
    ],
    embedding=embedder, collection_name="support",
)

store_map = {
    "product_docs" : product_store.as_retriever(search_kwargs={'k': 2}),
    "pricing_docs" : pricing_store.as_retriever(search_kwargs={'k': 2}),
    "support_docs" : support_store.as_retriever(search_kwargs={'k': 2}),
}

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a technical assistant for Datanous.ai
    Answer with ONLY the context below. If not found, say "I do not have this information.
    
    Context:
    {context}
    
    Question: {question}
    
    Answer:"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)
 
def routed_rag(question: str) -> str:
    """Route question -> select store -> retrieve -> generate."""
    decision  = router_chain.invoke({"question": question})
    retriever = store_map[decision.datasource]
    docs      = retriever.invoke(question)
    context   = format_docs(docs)
    return (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
 
# Test routed RAG
for q in [
    "What document formats does Datanous Insight support?",
    "How much does the Professional plan cost?",
    "Why is Datanous Guard blocking my queries with a 0.7 threshold?",
]:
    print(f"Q: {q}")
    print(f"A: {routed_rag(q)}\n")

Q: What document formats does Datanous Insight support?
A: Datanous Insight supports PDF, Word, Markdown, HTML, and plain-text formats.

Q: How much does the Professional plan cost?
A: 350 USD per month.

Q: Why is Datanous Guard blocking my queries with a 0.7 threshold?
A: Datanous Guard is blocking your queries because the grounding threshold is set to 0.7, which may be resulting in false positives. You can lower the threshold to 0.5 to reduce these false positives.



## 4.4 Query Construction — από φυσική γλώσσα σε δομημένα queries

Φαντάσου ότι έχεις μια vector store με metadata όπως:

```yaml
doc_id: "INC-2024-0431"
product: "orbit_storage"
severity: "critical"
region: "eu-west-1"
created_at: "2024-09-12"
resolved: true
duration_minutes: 47
```

Ένας χρήστης ρωτά: *«Δείξε critical incidents του Orbit Storage στην Ευρώπη μετά τον Μάιο 2024 που διήρκεσαν >30 λεπτά»*

Αυτή η ερώτηση δεν είναι semantic search — είναι **structured query**. Με embedding-only retrieval θα αποτυγχάναμε.

**Λύση:** Χρησιμοποιούμε LLM για να μετατρέψουμε την ερώτηση σε **structured filter** (Pydantic), και η vector store κάνει retrieval ΜΕ filter.

In [4]:
from datetime import date
from typing import Optional
 
class SupportTicketQuery(BaseModel):
    """Structured query for Datanous.ai support ticket database."""
 
    text_search: str = Field(...,description="Semantic search query for the ticket description",)
    product: Optional[Literal["Datanous Insight", "Datanous Search", "Datanous Guard", "Datanous Embed"]] = Field(None,description="Datanous.ai product name if explicitly mentioned",)
    severity: Optional[Literal["low", "medium", "high", "critical"]] = Field(None,description="Ticket severity if mentioned",)
    tenant_tier: Optional[Literal["Starter", "Professional", "Enterprise"]] = Field(None,description="Subscription tier of the affected tenant if mentioned",)
    created_after: Optional[date] = Field(None,description="Only tickets created after this date",)
    created_before: Optional[date] = Field(None,description="Only tickets created before this date",)
    resolved: Optional[bool] = Field(None,description="Filter by resolution status if mentioned",)
    min_duration_minutes: Optional[int] = Field(None,description="Only tickets whose duration is at least this many minutes",)
    max_duration_minutes: Optional[int] = Field(None,description="Only tickets whose duration is at most this many minutes",)
 
TICKET_QUERY_PROMPT = ChatPromptTemplate.from_template(
    """Convert the user request into a structured SupportTicketQuery for Datanous.ai.
Extract only what is explicitly mentioned.
 
User request: {question}"""
)
 
ticket_query_chain = (
    TICKET_QUERY_PROMPT
    | llm.with_structured_output(SupportTicketQuery)
)
 
# Test query construction
test_requests = [
    "Show me critical unresolved Datanous Guard issues from the last month",
    "Find high-severity Datanous Search tickets for Enterprise clients",
    "Any Datanous Embed errors reported after June 2024?",
    "Show tickets that lasted more than 30 minutes",
]
 
for req in test_requests:
    q = ticket_query_chain.invoke({"question": req})
    print(f"Request : {req}")
    print(f"  text_search : {q.text_search}")
    print(f"  product     : {q.product}")
    print(f"  severity    : {q.severity}")
    print(f"  tenant_tier : {q.tenant_tier}")
    print(f"  resolved    : {q.resolved}")
    print(f"  created_after : {q.created_after}")
    print(f"  min_duration : {q.min_duration_minutes}")
    print()

Request : Show me critical unresolved Datanous Guard issues from the last month
  text_search : 
  product     : Datanous Guard
  severity    : critical
  tenant_tier : None
  resolved    : False
  created_after : 2023-09-01
  min_duration : None

Request : Find high-severity Datanous Search tickets for Enterprise clients
  text_search : 
  product     : Datanous Search
  severity    : high
  tenant_tier : Enterprise
  resolved    : None
  created_after : None
  min_duration : None

Request : Any Datanous Embed errors reported after June 2024?
  text_search : Datanous Embed errors
  product     : Datanous Embed
  severity    : None
  tenant_tier : None
  resolved    : None
  created_after : 2024-06-01
  min_duration : None

Request : Show tickets that lasted more than 30 minutes
  text_search : 
  product     : None
  severity    : None
  tenant_tier : None
  resolved    : None
  created_after : None
  min_duration : 30



### Εκτέλεση structured query σε vector store

Τα filters του Chroma δέχονται operators όπως `$gte`, `$lte`, `$eq`, `$in`, `$and`.

In [5]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
 
# Synthetic support-ticket corpus with metadata.
# This is the vector store used by both the manual structured-query demo
# and the SelfQueryRetriever demo in the next section.
ticket_docs = [
    Document(
        page_content="Critical unresolved incident: Datanous Guard blocked legitimate tenant requests after a policy update.",
        metadata={
            "ticket_id": "INC-2024-0431",
            "product": "Datanous Guard",
            "severity": "critical",
            "tenant_tier": "Enterprise",
            "created_at": "2024-09-12",
            "resolved": False,
            "duration_minutes": 47,
        },
    ),
    Document(
        page_content="High severity Datanous Search issue: metadata filter syntax caused empty result sets for a Professional tenant.",
        metadata={
            "ticket_id": "INC-2024-0442",
            "product": "Datanous Search",
            "severity": "high",
            "tenant_tier": "Professional",
            "created_at": "2024-10-03",
            "resolved": True,
            "duration_minutes": 18,
        },
    ),
    Document(
        page_content="Datanous Embed returned 429 errors because the client exceeded the maximum batch size.",
        metadata={
            "ticket_id": "INC-2024-0450",
            "product": "Datanous Embed",
            "severity": "medium",
            "tenant_tier": "Starter",
            "created_at": "2024-10-21",
            "resolved": True,
            "duration_minutes": 22,
        },
    ),
    Document(
        page_content="Datanous Insight Professional tenant reported cross-tenant rows because tenant_id middleware was disabled.",
        metadata={
            "ticket_id": "INC-2024-0463",
            "product": "Datanous Insight",
            "severity": "critical",
            "tenant_tier": "Professional",
            "created_at": "2024-11-05",
            "resolved": False,
            "duration_minutes": 63,
        },
    ),
    Document(
        page_content="Low severity Datanous Search timeout affected an Enterprise tenant during a regional index rebuild.",
        metadata={
            "ticket_id": "INC-2024-0471",
            "product": "Datanous Search",
            "severity": "low",
            "tenant_tier": "Enterprise",
            "created_at": "2024-11-18",
            "resolved": True,
            "duration_minutes": 9,
        },
    ),
]
 
tickets_store = Chroma.from_documents(
    ticket_docs,
    embedding=embedder,
    collection_name="datanous_support_tickets",
)
 
 
def build_chroma_filter(q: SupportTicketQuery) -> dict | None:
    """Translate our Pydantic query object into a Chroma metadata filter."""
    clauses = []
 
    if q.product:
        clauses.append({"product": {"$eq": q.product}})
    if q.severity:
        clauses.append({"severity": {"$eq": q.severity}})
    if q.tenant_tier:
        clauses.append({"tenant_tier": {"$eq": q.tenant_tier}})
    if q.resolved is not None:
        clauses.append({"resolved": {"$eq": q.resolved}})
    if q.created_after:
        clauses.append({"created_at": {"$gte": q.created_after.isoformat()}})
    if q.created_before:
        clauses.append({"created_at": {"$lte": q.created_before.isoformat()}})
    if q.min_duration_minutes is not None:
        clauses.append({"duration_minutes": {"$gte": q.min_duration_minutes}})
    if q.max_duration_minutes is not None:
        clauses.append({"duration_minutes": {"$lte": q.max_duration_minutes}})
 
    if not clauses:
        return None
    if len(clauses) == 1:
        return clauses[0]
    return {"$and": clauses}
 
 
def run_ticket_search(question: str, k: int = 5):
    """Natural language -> structured query -> Chroma similarity search + metadata filter."""
    structured_query = ticket_query_chain.invoke({"question": question})
    chroma_filter = build_chroma_filter(structured_query)
 
    print(f"Question: {question}")
    print(f"Text search: {structured_query.text_search}")
    print(f"Chroma filter: {chroma_filter}\n")
 
    results = tickets_store.similarity_search(
        structured_query.text_search,
        k=k,
        filter=chroma_filter,
    )
 
    if not results:
        print("No matching tickets found.\n")
        return []
 
    for doc in results:
        m = doc.metadata
        print(
            f"  {m['ticket_id']} | {m['product']} | {m['severity']} | "
            f"tier={m['tenant_tier']} | resolved={m['resolved']} | "
            f"duration={m['duration_minutes']} min"
        )
        print(f"  {doc.page_content}\n")
 
    return results
 
 
# Test manual structured retrieval
for question in [
    "unresolved critical Datanous Guard tickets",
    "high severity Datanous Search issues for Professional clients",
    "any Datanous Embed tickets that lasted more than 15 minutes",
]:
    run_ticket_search(question)

Question: unresolved critical Datanous Guard tickets
Text search: unresolved
Chroma filter: {'$and': [{'product': {'$eq': 'Datanous Guard'}}, {'severity': {'$eq': 'critical'}}, {'resolved': {'$eq': False}}]}

  INC-2024-0431 | Datanous Guard | critical | tier=Enterprise | resolved=False | duration=47 min
  Critical unresolved incident: Datanous Guard blocked legitimate tenant requests after a policy update.

Question: high severity Datanous Search issues for Professional clients
Text search: 
Chroma filter: {'$and': [{'product': {'$eq': 'Datanous Search'}}, {'severity': {'$eq': 'high'}}, {'tenant_tier': {'$eq': 'Professional'}}]}

  INC-2024-0442 | Datanous Search | high | tier=Professional | resolved=True | duration=18 min
  High severity Datanous Search issue: metadata filter syntax caused empty result sets for a Professional tenant.

Question: any Datanous Embed tickets that lasted more than 15 minutes
Text search: 
Chroma filter: {'$and': [{'product': {'$eq': 'Datanous Embed'}}, {'

## 4.5 Self-Querying Retriever

Το LangChain παρέχει έτοιμο **`SelfQueryRetriever`** που κάνει αυτό αυτόματα: ορίζεις το **schema** και αυτός φτιάχνει το structured query.

In [ ]:
# Required once for SelfQueryRetriever's structured-query parser:
# %pip install -U lark langchain-classic
 
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo
 
metadata_field_info = [
    AttributeInfo(
        name="product",
        description="Datanous.ai product name. One of: Datanous Insight, Datanous Search, Datanous Guard, Datanous Embed.",
        type="string",
    ),
    AttributeInfo(
        name="severity",
        description="Ticket severity level. One of: low, medium, high, critical.",
        type="string",
    ),
    AttributeInfo(
        name="tenant_tier",
        description="Subscription tier. One of: Starter, Professional, Enterprise.",
        type="string",
    ),
    AttributeInfo(
        name="resolved",
        description="Whether the ticket is resolved.",
        type="boolean",
    ),
    AttributeInfo(
        name="duration_minutes",
        description="Duration of the issue in minutes.",
        type="integer",
    ),
]
 
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=tickets_store,
    document_contents="Datanous.ai support ticket description",
    metadata_field_info=metadata_field_info,
    enable_limit=True,
)
 
# Natural language filter queries
test_queries = [
    "unresolved critical Datanous Guard tickets",
    "high severity Datanous Search issues for Professional clients",
    "any Datanous Embed tickets that lasted more than 15 minutes",
]
 
for q in test_queries:
    print(f"Query: {q}")
    results = self_query_retriever.invoke(q)
 
    if not results:
        print("  No results found.")
 
    for r in results:
        m = r.metadata
        print(
            f"  [{m.get('product')} | {m.get('severity')} | "
            f"tier={m.get('tenant_tier')} | resolved={m.get('resolved')} | "
            f"duration={m.get('duration_minutes')}] {r.page_content[:90]}..."
        )
    print()

Query: unresolved critical Datanous Guard tickets
  [Datanous Guard | critical | tier=Enterprise | resolved=False | duration=47] Critical unresolved incident: Datanous Guard blocked legitimate tenant requests after a po
  [Datanous Insight | critical | tier=Professional | resolved=False | duration=63] Datanous Insight Professional tenant reported cross-tenant rows because tenant_id middlewa

Query: high severity Datanous Search issues for Professional clients
  [Datanous Search | high | tier=Professional | resolved=True | duration=18] High severity Datanous Search issue: metadata filter syntax caused empty result sets for a

Query: any Datanous Embed tickets that lasted more than 15 minutes
  [Datanous Embed | medium | tier=Starter | resolved=True | duration=22] Datanous Embed returned 429 errors because the client exceeded the maximum batch size.



## 4.6 Routing patterns — best practices

| Pattern | Πότε ταιριάζει | Πλεονέκτημα |
|---|---|---|
| **LLM-based logical routing** | Πολλές ξεχωριστές πηγές | Ευέλικτο, μπορεί να εξηγήσει την επιλογή |
| **Embedding-based semantic routing** | Επιλογή prompt/persona | Γρήγορο, χωρίς LLM call |
| **Rule-based / regex** | Σαφή keywords (πχ «PRICING» στην ερώτηση) | Deterministic, μηδαμινό cost |
| **Hybrid (rules → LLM fallback)** | Production-ready | Βέλτιστος συνδυασμός cost/accuracy |

**Pro tip:** Σε production συνδύασε rules-first για 80% των cases (zero LLM cost) και LLM router για το υπόλοιπο 20%.

## 4.7 Άσκηση

**Άσκηση:** Φτιάξε ένα `HybridRouter` class που:

1. Πρώτα ψάχνει για keyword patterns (πχ «cost», «τιμή», «$» → pricing)
2. Αν δεν match-άρει, καλεί τον LLM router
3. Επιστρέφει το όνομα του datasource

Δοκιμάσε τον σε διάφορες ερωτήσεις και μέτρα πόσες πιάστηκαν από τους κανόνες vs LLM.

## 📋 Συμπεράσματα

| # | Έννοια | Συνοπτικά |
|---|---|---|
| 1 | Πρόβλημα | Πολλές πηγές γνώσης / πολλοί τύποι ερωτήσεων |
| 2 | Logical routing | LLM επιστρέφει ποιο datasource, με `with_structured_output()` |
| 3 | Pydantic Literal | Περιορίζει τις επιτρεπτές απαντήσεις σε enum |
| 4 | Semantic routing | Embedding similarity με template prompts → επιλογή persona |
| 5 | Query construction | NL → structured Pydantic query με metadata filters |
| 6 | Vector + filter | Συνδυασμός semantic search + structured filtering |
| 7 | SelfQueryRetriever | Έτοιμο abstraction του LangChain |
| 8 | Hybrid router | Rules-first + LLM fallback = cost-optimal |
| 9 | Επόμενο βήμα | Notebook 05 — Advanced Indexing |